# Plot for comparison on fev-bench datasets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
# Metrics per (family, zone) for (SigAdapt, Chronos-2, TiDE), read from results/
# (run scripts/make_tables.py first)
import pandas as pd
from pathlib import Path

tab = pd.read_csv(Path.cwd().parent / "results" / "summary_per_series.csv", dtype={"series": str})
families = {"gfc12": "GFC12", "gfc14": "GFC14", "gfc17": "GFC17", "entsoe": "ENTSO-E"}


def table(metric):
    out = {}
    for d, family in families.items():
        t = tab[tab["dataset"] == d].pivot(index="series", columns="model", values=metric)
        t = t[["sigadapt", "chronos2", "tide"]].dropna()
        t = t.loc[sorted(t.index, key=lambda z: (len(z), z))]
        if d == "gfc14":
            t.index = ["-"]
        out[family] = {z: tuple(r) for z, r in t.iterrows()}
    return out


rmse, mape = table("RMSE"), table("MAPE")

colors  = {"GFC12": "tab:blue", "GFC14": "tab:orange",
           "GFC17": "tab:green", "ENTSO-E": "tab:red"}
markers = {"Chronos-2": "o", "TiDE": "s"}

h_fam = [Line2D([], [], marker="o", ls="", color=c, mec="black", mew=0.5, label=f)
         for f, c in colors.items()]
h_mod = [Line2D([], [], marker=m, ls="", color="white", mec="black", label=n)
         for n, m in markers.items()]


def best_competitor(chronos, tide):
    return (chronos, "Chronos-2") if chronos <= tide else (tide, "TiDE")




# Plot 2: relative RMSE gain of SigAdapt over the best competitor
# gain = (best_comp - sig) / best_comp      (> 0 means SigAdapt is better)

fig, ax = plt.subplots(figsize=(10, 4))

x = 0
ticks, ticklabels, family_centres = [], [], []

for family, zones in rmse.items():
    start = x
    for zone, (sig, chronos, tide) in zones.items():
        best, name = best_competitor(chronos, tide)
        gain = 100 * (best - sig) / best
        ax.scatter(x, gain, color=colors[family], marker=markers[name],
                   s=45, edgecolor="black", linewidth=0.5, zorder=3)
        ticks.append(x)
        ticklabels.append("" if zone == "-" else zone)
        x += 1
    family_centres.append(((start + x - 1) / 2, family))
    x += 1  # empty slot between families

ax.axhline(0, color="black", linestyle="--", linewidth=1, zorder=1)
ax.axhspan(0, 22, color="lightgreen", alpha=0.25, zorder=0)   # SigAdapt-better region

ax.set_xticks(ticks)
ax.set_xticklabels(ticklabels, rotation=45, ha="right", fontsize=11)
ax.set_xlim(-1, x - 1)
ax.set_ylim(-15, 22)
ax.set_ylabel("RMSE gain of SigAdapt over\nbest competitor (%)")

for centre, family in family_centres:
    ax.text(centre, 21.3, family, ha="center", va="top", fontsize=11,
            color=colors[family], fontweight="bold")

ax.legend(handles=h_mod, title="Best competitor", loc="lower left", fontsize=8)
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
fig_dir = Path.cwd().parent / "figures"
fig_dir.mkdir(exist_ok=True)
fig.savefig(fig_dir / "rmse_gain.pdf")
fig.savefig(fig_dir / "rmse_gain.png", dpi=200)
